In [2]:
# ============================================================
# Setup: imports + synthetic FinalData (원본 .mat 미제공 대체)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

plt.close("all")
np.random.seed(42)

# ---- Problem 1 parameters ----
p1_t_s = 0.0
p1_t_e = 10.0
p1_dt  = 0.02
p1_Q       = np.array([[1e-5, 0.0],
                       [0.0 , 5e-4]], dtype=float)
p1_X_init  = np.array([[1.0],
                       [0.0]], dtype=float)

# ---- Problem 2 parameters ----
p2_t_e = 15.0
p2_dt  = 0.05
_p2_t  = np.arange(0.0, p2_t_e + p2_dt, p2_dt)
p2_imu = np.vstack([
    0.8*np.sin(0.35*_p2_t) + 0.15*np.cos(1.1*_p2_t),     # acceleration
    0.18*np.cos(0.22*_p2_t) + 0.03*np.sin(1.5*_p2_t),    # yaw rate
])
p2_Q   = np.diag([5e-4, 5e-4, 1e-3, 2e-4]).astype(float)

# ---- Problem 3 parameters ----
p3_time          = np.arange(0.5, 10.5, 0.5)
_true_gravity    = 3.711                                 # 외계 중력 (실제값, 채점 비교용)
p3_meas_noise    = 0.15 + 0.03*p3_time                   # std (시간에 따라 다름)
p3_meas_range    = 0.5*_true_gravity*(p3_time**2) + np.random.normal(0.0, p3_meas_noise)
p3_init_gravity  = np.array([[8.0]],  dtype=float)
p3_init_P        = np.array([[25.0]], dtype=float)

# ---- Problem 4 parameters ----
p4_t_e = 20.0
p4_dt  = 0.05
_p4_time = np.arange(0.0, p4_t_e + p4_dt, p4_dt)
p4_Q     = np.array([[1e-5, 0.0],
                     [0.0 , 1e-3]], dtype=float)
p4_R     = np.array([[0.35**2]], dtype=float)
p4_X_init = np.array([[0.5],
                      [0.0]], dtype=float)
p4_P_init = np.array([[0.2, 0.0],
                      [0.0, 0.2]], dtype=float)
p4_force  = 250*np.sin(0.7*_p4_time) + 100*np.cos(0.13*_p4_time)
_k4, _b4, _m4 = 1000.0, 200.0, 800.0
_F4 = np.array([[1.0, p4_dt],
                [-(_k4*p4_dt)/_m4, 1.0 - (_b4*p4_dt)/_m4]], dtype=float)
_B4 = np.array([[0.0], [p4_dt/_m4]], dtype=float)
_H4 = np.array([[3.0, 1.0]], dtype=float)
_true_x4 = np.zeros((2, len(_p4_time)))
_true_x4[:, 0] = np.squeeze(p4_X_init)
for _k in range(1, len(_p4_time)):
    _w = np.random.multivariate_normal(np.zeros(2), p4_Q)
    _true_x4[:, _k] = _F4 @ _true_x4[:, _k-1] + (_B4[:, 0] * p4_force[_k-1]) + _w
p4_meas   = (_H4 @ _true_x4).reshape(-1) + np.random.normal(0.0, np.sqrt(p4_R[0,0]), size=len(_p4_time))
p4_true_x = _true_x4[0, :]                  # 비교용 true position

# ---- Problem 5 parameters (rocket) ----
p5_t_e = 30.0
p5_dt  = 0.1
_p5_time = np.arange(0.0, p5_t_e + p5_dt, p5_dt)
p5_Q     = np.diag([1e-3, 1e-3, 2e-2, 2e-2, 5e-4, 1e-1]).astype(float)
p5_R     = np.diag([4.0, 4.0, 1.0, 1.0]).astype(float)
p5_X_init = np.array([[0.0], [0.0], [0.0], [0.0], [np.pi/2], [1000.0]], dtype=float)
p5_P_init = np.diag([1.0, 1.0, 1.0, 1.0, 0.05, 10.0]).astype(float)
_thrust    = 14000 + 2500*np.sin(0.18*_p5_time) + 1500*np.cos(0.05*_p5_time)
_yaw_rate  = 0.012*np.sin(0.4*_p5_time) - 0.01*np.exp(-0.08*_p5_time)
p5_input   = np.vstack([_thrust, _yaw_rate])

_g = np.array([0.0, -10.0])
_true_x5 = np.zeros((6, len(_p5_time)))
_true_x5[:, 0] = np.squeeze(p5_X_init)
for _k in range(1, len(_p5_time)):
    px, py, vx, vy, yaw, mass = _true_x5[:, _k-1]
    th = p5_input[0, _k-1]; yr = p5_input[1, _k-1]
    mass = max(mass, 50.0)
    acc  = np.array([(th/mass)*np.cos(yaw), (th/mass)*np.sin(yaw)]) + _g
    md_  = -th/1000.0
    pn   = np.random.multivariate_normal(np.zeros(6), p5_Q)
    _true_x5[:, _k] = np.array([
        px + vx*p5_dt,
        py + vy*p5_dt,
        vx + acc[0]*p5_dt,
        vy + acc[1]*p5_dt,
        yaw + yr*p5_dt,
        max(50.0, mass + md_*p5_dt),
    ]) + pn
    _true_x5[5, _k] = max(50.0, _true_x5[5, _k])
_H5 = np.array([[1,0,0,0,0,0],
                [0,1,0,0,0,0],
                [0,0,1,0,0,0],
                [0,0,0,1,0,0]], dtype=float)
p5_meas = _H5 @ _true_x5 + np.random.multivariate_normal(np.zeros(4), p5_R, size=len(_p5_time)).T

print("Synthetic data ready.")


Synthetic data ready.


In [19]:
k, b, m = 1000.0, 300.0, 300.0

dt = p1_dt
time = np.arange(dt, p1_t_e+dt, dt)
n = len(time)


# 시스템 행렬 
F = np.array([[1, dt],
             [-(k/m)*dt, 1-(b/m)*dt]])

X = np.zeros((2,n))
X[:,0] = p1_X_init.flatten() # 1차원으로 맞춰줌 
P = [np.zeros((2,2)) for _ in range(n)]
P[0] = np.zeros((2,2))

for k in range(1,n):
    X[:,k] = F @ X[:,k-1]
    P[k] = F @ P[k-1] @ F.T + p1_Q
    
print("F",F)
print("\nFinal state", X[:,-1])
print("\ncov",P[-1])


F [[ 1.          0.02      ]
 [-0.06666667  0.98      ]]

Final state [0.00129665 0.01577785]

cov [[ 0.00436432 -0.00039387]
 [-0.00039387  0.01441521]]


In [18]:
k, b, m = 1000.0, 300.0, 300.0

dt = p1_dt
time = np.arange(dt,p1_t_e+dt,dt)
n = len(time)

# 시스템 행렬 
F = np.array([[1, dt],
              [-(k/m)*dt, 1-(b/m)*dt]])

# 상태 공분산 바구니, 초기화  
X = np.zeros((2,n))
X[:,0] = p1_X_init.flatten()

P = [np.zeros((2,2)) for _ in range(n)]
P[0] = np.zeros((2,2))

for k in range(1,n):
    X[:,k] = F @ X[:,k-1]
    P[k] = F @ P[k-1] @ F.T + p1_Q
    
print(X[:,-1])






[0.00129665 0.01577785]


In [ ]:
k, b, m = 1000.0, 300.0, 300.0

dt = p1_dt
time = np.arange(dt, dt+ p1_t_e, dt)
n = len(time)

# F 
F = np.array([[1, dt],
              [-(k/m)*dt, 1-(b/m)*dt]])

# 상태 공분산 바구니, 초기화 
X = np.zeros((2,n))  ## 실수 주의 
X[:,0] = p1_X_init.flatten()

P = [np.zeros((2,2)) for _ in range(n)]
P[0] = np.zeros((2,2))



# 상태, 공분산 전파 
for k in range(1,n):
    X[:, k] = F @ X[:, k-1] 
    P[k] = F @ P[k-1] @ F.T + p1_Q


In [28]:
k, b, m = 1000.0, 300.0, 300.0

dt = p1_dt
time = np.arange(dt, dt + p1_t_e, dt)
n = len(time)

# F 
F = np.array([[1, dt],
              [-(k/m)*dt, 1-(b/m)*dt]])

# state cov 바구니 초기화 
X = np.zeros((2,n))
X[:, 0] = p1_X_init.flatten()

P = [np.zeros((2,2)) for _ in range(n)]
P[0] = np.zeros((2,2))

# 전파 
for k in range(1,n):
    X[:,k] = F @ X[:,k-1] 
    P[k] = F @ P[k-1] @ F.T + p1_Q
    
    
    
    


In [35]:
dt = float(p2_dt)
time = np.arange(dt, p2_t_e+dt,dt)
n = len(time)

# F , X , P 
F_list = [np.eye(4) for _ in range(n)]

X = np.zeros((4,n))
X[:, 0] = np.array([0,0,0,1])

P = [np.zeros((4,4)) for _ in range(n)]

for k in range(1,n):
    # 초기값 설정 
    x, y, v, yaw = X[:,k-1]
    a = p2_imu[0,k-1]
    yaw_rate = p2_imu[1,k-1]
    
    c, s = np.cos(yaw), np.sin(yaw)
    
    # 비선형 상태 전파 
    X[:, k] = np.array([
        x + v*c*dt,
        y + v*s*dt,
        v + a*dt,
        yaw + yaw_rate * dt
    ])
        
    
    # 선형화 F 
    F = np.eye(4)
    F[0,2] = c*dt
    F[0,3] = -s*v*dt
    F[1,2] = s*dt
    F[1,3] = v*c*dt 
    F_list[k] = F
    
    # 공분산 전파 
    P[k] = F @ P[k-1] @ F.T + p2_Q
    

print(X[:,-1])

[-2.24957545 38.9332488   1.08199428  0.92722716]


In [ ]:
dt = float(p2_dt)
time = np.arange(dt, p2_t_e+dt,dt)
n = len(time)

# F X P  
X = np.zeros((4,n))
X[:,0] = np.array([0,0,0,1])

P = [np.zeros((4,4)) for _ in range(n)]


# 상태 공분산 전파 
for k in range(1,n):
    # 초기값 설정 
    x,y,v,yaw = X[:,k-1]
    a = p2_imu[0,k-1]
    yaw_rate = p2_imu[1,k-1]
    
    c,s = np.cos(yaw), np.sin(yaw)
    
    # 비선형 전파 
    X[: ,k] = np.array([
        x + v*c*dt,
        y + v*s*dt,
        v + a*dt,
        yaw + yaw_rate * dt
    ]) 
    
    # 자코비안 F 
    F = np.eye(4)
    F[0,2] = c*dt
    F[0,3] = v*s*dt
    F[1,2] = s *dt
    F[1,3] = v*c*dt
    
    # 공분산 전파 
    P[k] = F @ P[k-1] @ F.T + p2_Q


print(X[:,-1])


[-2.24957545 38.9332488   1.08199428  0.92722716]


In [ ]:
#####################################################################
# 문제 3: Least Square / Recursive Least Square
time  = np.array(p3_time)
y     = np.array(p3_meas_range)
sigma = np.array(p3_meas_noise)
n_step = len(time)

# placeholders
a3_LS_H  = np.zeros((n_step, 1))

# ============================================================
# 1) Least Square (batch)
# ============================================================
a3_LS_H[:, 0] = 0.5 * time**2
a3_LS_X = np.linalg.inv(a3_LS_H.T @ a3_LS_H) @ a3_LS_H.T @ y

# ============================================================
# 2) Recursive Least Square (sequential)
# ============================================================

# 초기화 
x_hat = np.array(p3_init_gravity)
P     = np.array(p3_init_P)
I     = np.eye(1)

for i in range(n_step):
    H = np.array([[0.5 * time[i]**2]])
    R = np.array([[sigma[i]**2]])
    z = np.array([[y[i]]])

    # Kalman gain
    K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)

    # state / covariance update
    x_hat = x_hat + K @ (z - H @ x_hat)
    P     = (I - K @ H) @ P


print(a3_LS_X)
print(x_hat)

a3_LS_X (LS gravity)      = 3.702214440552611
[3.70221444]
[[3.70299629]]


In [45]:

n_state = 2
dt      = float(p4_dt)
time    = np.arange(0.0, p4_t_e + dt, dt)
n_step  = len(time)

k_, b_, m_ = 1000.0, 200.0, 800.0

# 시스템 행렬 (시불변이므로 한 번만 정의)
a4_F = np.array([[1.0,           dt          ],
                 [-(k_/m_)*dt,   1 - (b_/m_)*dt]])
a4_H = np.array([[3.0, 1.0]])
B    = np.array([[0.0],
                 [dt/m_]])
I    = np.eye(n_state)

# 결과 저장 그릇
a4_X      = np.zeros((n_state, n_step))
a4_X[:,0] = p4_X_init.flatten()

a4_P      = [None]*n_step # 리스트 
a4_P[0]   = np.array(p4_P_init)

# Kalman Filter 루프
for k in range(1, n_step):
    u = p4_force[k-1]                # scalar 입력
    z = p4_meas[k]                   # scalar 측정값

    # ---- Predict ----
    x_pri = a4_F @ a4_X[:, k-1] + B@u.flatten()#(B * u).flatten()
    P_pri = a4_F @ a4_P[k-1] @ a4_F.T + p4_Q

    # ---- Update ----
    S = a4_H @ P_pri @ a4_H.T + p4_R
    K = P_pri @ a4_H.T @ np.linalg.inv(S)

    a4_X[:, k] = x_pri + (K @ (z - a4_H @ x_pri)).flatten()
    a4_P[k]    = (I - K @ a4_H) @ P_pri

print("a4_H =", a4_H)
print("a4_F =\n", a4_F)
print("Final posterior state =", a4_X[:, -1])

a4_H = [[3. 1.]]
a4_F =
 [[ 1.      0.05  ]
 [-0.0625  0.9875]]
Final posterior state = [0.40881554 0.5886297 ]


In [ ]:

dt      = float(p5_dt)
time    = np.arange(0.0, p5_t_e + dt, dt)
n_step  = len(time)
n_state = 6

# ---- 측정 행렬 (선형) ----
H = np.array([[1,0,0,0,0,0],
              [0,1,0,0,0,0],
              [0,0,1,0,0,0],
              [0,0,0,1,0,0]], dtype=float)
I = np.eye(n_state)

# ---- 결과 그릇 ----
a5_X      = np.zeros((n_state, n_step))
a5_X[:,0] = p5_X_init.flatten()

a5_P      = [None]*n_step
a5_P[0]   = np.array(p5_P_init)


# ---- 비선형 모델 함수 ----
def rocket_f(x, u, dt):
    px, py, vx, vy, yaw, mass = x
    T, yaw_rate = u
    mass = max(mass, 50.0)              # 연료 고갈 방지
    return np.array([
        px + vx*dt,
        py + vy*dt,
        vx + (T/mass)*np.cos(yaw)*dt,
        vy + ((T/mass)*np.sin(yaw) - 10.0)*dt,
        yaw + yaw_rate*dt,
        mass - (T/1000.0)*dt
    ])

# ---- EKF 루프 ----
for k in range(1, n_step):
    x_prev = a5_X[:, k-1]
    u      = p5_input[:, k-1]
    z      = p5_meas[:, k]

    # ===== Predict =====
    # (1) 상태: 비선형 모델 그대로
    x_pri = rocket_f(x_prev, u, dt)

    # (2) 자코비안 F (이전 상태 기준으로 선형화)
    px, py, vx, vy, yaw, mass = x_prev
    T, yaw_rate = u
    mass = max(mass, 50.0)
    c, s = np.cos(yaw), np.sin(yaw)

    F = np.eye(n_state)
    F[0, 2] = dt
    F[1, 3] = dt
    F[2, 4] = -(T/mass)*s*dt
    F[2, 5] = -(T/mass**2)*c*dt
    F[3, 4] =  (T/mass)*c*dt
    F[3, 5] = -(T/mass**2)*s*dt

    # (3) 공분산
    P_pri = F @ a5_P[k-1] @ F.T + p5_Q

    # ===== Update =====
    S = H @ P_pri @ H.T + p5_R           # innovation covariance
    K = P_pri @ H.T @ np.linalg.inv(S)   # Kalman gain

    a5_X[:, k] = x_pri + K @ (z - H @ x_pri)
    a5_P[k]    = (I - K @ H) @ P_pri

print("Final state =", a5_X[:, -1])

Final state = [6.33755030e+02 4.21671927e+03 7.80121457e+01 2.92000112e+02
 1.26928910e+00 5.50456085e+02]
